# Research · Single-Image Dehazing — Dark Channel Prior pipeline (NB 1/2)

**Group 01 · Dept. of CSE, East West University** — Md. Asif Hossain (2022-3-60-007) · Nabil Subhan
(2022-3-60-063) · K M Nudar (2022-3-60-234)

Implements the classical **Dark Channel Prior** (He, Sun & Tang, CVPR 2009) end-to-end and visualises every stage.
The ablation study + quantitative tables live in **NB 2 (`dehaze-ablation-eval`)**. Attach a RESIDE/O-HAZE dataset
for real numbers; otherwise a **synthetic-haze fallback** runs automatically so the notebook works anywhere.

In [ ]:
# research-figure style + palette (colourblind-safe)
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 160, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "grid.alpha": 0.25, "axes.titleweight": "bold", "figure.autolayout": True})
PAL = {"blue": "#2f6db5", "orange": "#e07b39", "green": "#3f9b6d", "red": "#c0392b",
       "grey": "#7f8c8d", "purple": "#7d5ba6"}

In [ ]:
# ===== Dark Channel Prior — classical dehazing (pure NumPy/OpenCV) =====
import numpy as np, cv2

def dark_channel(img, patch=15):
    mn = img.min(axis=2)
    k = cv2.getStructuringElement(cv2.MORPH_RECT, (patch, patch))
    return cv2.erode(mn, k)

def atmospheric_light(img, dark, method="dcp_top"):
    h, w = dark.shape; flat_img = img.reshape(-1, 3)
    if method == "brightest":
        return flat_img[img.reshape(-1, 3).sum(1).argmax()]
    if method == "quadtree":
        reg = img
        while reg.shape[0] * reg.shape[1] > 400:
            H, W = reg.shape[:2]; hh, ww = H // 2, W // 2
            quads = [reg[:hh, :ww], reg[:hh, ww:], reg[hh:, :ww], reg[hh:, ww:]]
            scores = [q.reshape(-1, 3).mean() - q.reshape(-1, 3).std() for q in quads]
            reg = quads[int(np.argmax(scores))]
            if min(reg.shape[:2]) < 2: break
        fr = reg.reshape(-1, 3); return fr[fr.sum(1).argmax()]
    n = max(1, int(0.001 * h * w))                       # dcp_top: top-0.1% dark-channel pixels
    idx = np.argpartition(dark.ravel(), -n)[-n:]
    cand = flat_img[idx]; return cand[cand.sum(1).argmax()]

def guided_filter(guide, src, r=60, eps=1e-3):
    r = max(1, int(r)); box = lambda x: cv2.boxFilter(x, cv2.CV_64F, (r, r))
    mI, mp = box(guide), box(src); mIp = box(guide * src)
    cov = mIp - mI * mp; var = box(guide * guide) - mI * mI
    a = cov / (var + eps); b = mp - a * mI
    return box(a) * guide + box(b)

def transmission(img, A, omega=0.95, patch=15):
    return 1.0 - omega * dark_channel(img / np.maximum(A, 1e-6), patch)

def recover(img, A, t, t0=0.1):
    tt = np.clip(t, t0, 1.0)[..., None]
    return np.clip((img - A) / tt + A, 0.0, 1.0)

def dehaze(img, patch=15, omega=0.95, t0=0.1, refine="guided", A_method="dcp_top", r=60, eps=1e-3):
    dark = dark_channel(img, patch)
    A = atmospheric_light(img, dark, A_method)
    t_raw = transmission(img, A, omega, patch)
    if refine == "guided":
        gray = cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float64) / 255.0
        t = guided_filter(gray, t_raw, r, eps)
    else:
        t = t_raw
    J = recover(img, A, t, t0)
    return {"dehazed": J, "dark": dark, "A": A, "t_raw": t_raw, "t": t}

In [ ]:
# ===== paired data: RESIDE/O-HAZE if attached, else synthetic-haze fallback =====
from pathlib import Path
from PIL import Image
import numpy as np

def _load(p, size=256):
    im = Image.open(p).convert("RGB").resize((size, size))
    return np.asarray(im, np.float64) / 255.0

def find_reside(maxn=60):
    root = Path("/kaggle/input")
    hazy_dir = next((d for d in root.rglob("*") if d.is_dir() and d.name.lower() in ("hazy", "haze")
                     and (any(d.glob("*.png")) or any(d.glob("*.jpg")))), None)
    if hazy_dir is None: return None
    gt_dir = next((d for d in hazy_dir.parent.iterdir() if d.is_dir() and
                   d.name.lower() in ("gt", "clear", "clean")), None)
    if gt_dir is None: return None
    gtmap = {}
    for g in list(gt_dir.glob("*.png")) + list(gt_dir.glob("*.jpg")):
        gtmap.setdefault("".join(ch for ch in g.stem if ch.isdigit()), g)
    pairs = []
    for hz in sorted(list(hazy_dir.glob("*.png")) + list(hazy_dir.glob("*.jpg"))):
        key = "".join(ch for ch in hz.stem if ch.isdigit()).split("_")[0] if hz.stem else ""
        key = "".join(ch for ch in hz.stem if ch.isdigit())
        g = gtmap.get(key) or next((v for k, v in gtmap.items() if key.startswith(k)), None)
        if g is not None:
            pairs.append((_load(hz), _load(g), "reside"))
        if len(pairs) >= maxn: break
    return pairs or None

def synth_haze(clear, beta, A_val):
    h, w = clear.shape[:2]
    yy, xx = np.mgrid[0:h, 0:w]
    depth = 0.2 + 0.8 * np.sqrt(((xx - w/2)/w)**2 + ((yy - h/2)/h)**2)   # radial depth proxy
    t = np.exp(-beta * depth)[..., None]
    A = np.array(A_val).reshape(1, 1, 3)
    hazy = clear * t + A * (1 - t)
    return np.clip(hazy, 0, 1)

def synthetic_bank():
    from skimage import data as skd
    clears = []
    for name in ["astronaut", "coffee", "chelsea", "rocket", "logo"]:
        try:
            im = getattr(skd, name)()
            if im.ndim == 3 and im.shape[2] >= 3:
                clears.append(_load_arr(im))
        except Exception:
            pass
    pairs, DENS = [], {0.8: "light", 1.6: "medium", 2.6: "dense"}
    for c in clears:
        for beta in (0.8, 1.6, 2.6):
            for A in ([0.85, 0.88, 0.9], [0.75, 0.78, 0.82]):
                pairs.append((synth_haze(c, beta, A), c, DENS[beta]))
    return pairs

def _load_arr(im, size=256):
    from PIL import Image as I
    return np.asarray(I.fromarray(im[..., :3]).convert("RGB").resize((size, size)), np.float64) / 255.0

pairs = find_reside()
SOURCE = "RESIDE/O-HAZE (attached)" if pairs else "synthetic-haze fallback"
if not pairs:
    pairs = synthetic_bank()
print(f"eval source: {SOURCE} | {len(pairs)} paired (hazy, clear) images")

In [ ]:
# ===== full-reference metrics =====
from skimage.metrics import peak_signal_noise_ratio as _psnr, structural_similarity as _ssim
def scores(dehazed, gt):
    return (_psnr(gt, dehazed, data_range=1.0),
            _ssim(gt, dehazed, data_range=1.0, channel_axis=2))

## Stage-by-stage visualisation
Haze model: `I = J·t + A·(1−t)`. DCP estimates `A`, the transmission `t`, refines `t` (guided filter), then
inverts the model to recover the clear radiance `J`.

In [ ]:
# stage figure on one representative hazy image
hazy, gt, _ = pairs[len(pairs)//2]
o = dehaze(hazy, patch=15, omega=0.95, t0=0.1, refine="guided", A_method="dcp_top")
p, s = scores(o["dehazed"], gt)
panels = [(hazy, "hazy input"), (o["dark"], "dark channel"), (o["t_raw"], "transmission (raw)"),
          (o["t"], "transmission (guided-refined)"), (o["dehazed"], f"recovered  PSNR {p:.1f} / SSIM {s:.2f}"),
          (gt, "ground truth")]
fig, ax = plt.subplots(2, 3, figsize=(13, 8))
for a, (im, t) in zip(ax.ravel(), panels):
    a.imshow(im, cmap="gray" if im.ndim == 2 else None, vmin=0, vmax=1); a.set_title(t, fontsize=9); a.axis("off")
plt.suptitle(f"Dark Channel Prior pipeline  (A = {np.round(o['A'],3).tolist()})", y=1.0)
plt.savefig("dcp_pipeline_stages.png", bbox_inches="tight"); plt.show()

## Sanity — a few dehazed examples with PSNR/SSIM vs the hazy baseline

In [ ]:
# baseline (no dehaze) vs DCP, on a handful of images
import numpy as np, pandas as pd
rows = []
for i in range(min(6, len(pairs))):
    hz, gt, dens = pairs[i]
    dp, ds = scores(hz, gt)                       # hazy vs GT (baseline)
    o = dehaze(hz); jp, js = scores(o["dehazed"], gt)
    rows.append({"img": i, "density": dens, "PSNR_hazy": dp, "PSNR_DCP": jp,
                 "SSIM_hazy": ds, "SSIM_DCP": js, "dPSNR": jp - dp})
san = pd.DataFrame(rows).round(3); print(san.to_string(index=False))
print(f"\nmean PSNR gain over hazy: {san.dPSNR.mean():+.2f} dB   (source: {SOURCE})")

fig, ax = plt.subplots(min(3, len(pairs)), 3, figsize=(11, 3.2*min(3, len(pairs))))
ax = np.atleast_2d(ax)
for r in range(ax.shape[0]):
    hz, gt, _ = pairs[r]; o = dehaze(hz)
    for c, (im, t) in enumerate([(hz, "hazy"), (o["dehazed"], "DCP"), (gt, "GT")]):
        ax[r, c].imshow(im, vmin=0, vmax=1); ax[r, c].set_title(t, fontsize=8); ax[r, c].axis("off")
plt.suptitle("DCP dehazing — qualitative sanity", y=1.0)
plt.savefig("dcp_sanity.png", bbox_inches="tight"); plt.show()

**Next:** `dehaze-ablation-eval` sweeps patch size, `A`-estimation, ω, `t₀`, and refinement, stratifies by
haze density, and quantifies DCP's sky/bright-region failure — the research contribution.